# RAGWire Setup and First Retrieval

**Stack:** Ollama · Qdrant (local)

Progressive journey:
- Ingest a single document and retrieve chunks
- Scale to multiple companies
- Explore metadata and add filters
- Build an agent — simple first, then filter-aware

## 1. Setup

In [11]:
import os
from dotenv import load_dotenv
load_dotenv()

# Route OpenAI-compatible clients (used by RAGWire's metadata LLM) to OpenRouter
os.environ["OPENAI_API_KEY"] = os.environ["OPENROUTER_API_KEY"]
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

from ragwire import RAGWire, setup_logging
import ragwire

print(ragwire.__version__)


1.2.9


In [12]:
logger = setup_logging(log_level="INFO")

## 2. Ingest a Single Document and Retrieve

**Model:** `qwen3.5:9b` · **Embedding:** `qwen3-embedding:0.6b` · **Vector Store:** Qdrant (local)

In [13]:
rag = RAGWire('5config_openrouter_qdrant.yaml')

2026-06-04 00:18:40,937 - ragwire.core.pipeline - INFO - Loading configuration from 5config_openrouter_qdrant.yaml
2026-06-04 00:18:40,996 - ragwire.core.pipeline - INFO - Document loader initialized
2026-06-04 00:18:40,997 - ragwire.core.pipeline - INFO - Text splitter initialized (strategy=markdown, chunk_size=10000)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 48619.16it/s]


2026-06-04 00:18:48,768 - ragwire.core.pipeline - INFO - Embedding model initialized (provider=huggingface)
2026-06-04 00:18:48,780 - ragwire.core.pipeline - INFO - Metadata extractor loaded from: finance_metadata.yaml
2026-06-04 00:18:48,780 - ragwire.core.pipeline - INFO - LLM initialized for metadata extraction (provider=openai, model=nvidia/nemotron-3-super-120b-a12b:free)
2026-06-04 00:18:48,802 - ragwire.vectorstores.qdrant_store - INFO - Connected to Qdrant at https://9fc30746-5851-4ae5-a4fb-9f27e6eacd42.eu-west-2-0.aws.cloud.qdrant.io:6333 (timeout=300s)
2026-06-04 00:18:49,361 - ragwire.core.pipeline - INFO - Using existing collection: finance-rag-qdrant
2026-06-04 00:18:53,148 - ragwire.core.pipeline - INFO - Vector store initialized
2026-06-04 00:18:53,150 - ragwire.core.pipeline - INFO - Retriever initialized (type=hybrid, top_k=5, auto_filter=False)
2026-06-04 00:18:53,150 - ragwire.core.pipeline - INFO - RAG pipeline initialized successfully


# RAGWire and Agentic RAG
- Upload set of documents
- Retrieval 
- Agentic RAG

## 3. Scale to Multiple Companies

Ingest all three 10-K filings at once. RAGWire deduplicates — re-running skips already-ingested files.

In [4]:
rag.ingest_directory('../data/finance_data')

2026-06-03 22:29:23,282 - ragwire.core.pipeline - INFO - Found 6 file(s) in ../data/finance_data
2026-06-03 22:29:23,286 - ragwire.core.pipeline - INFO - Starting ingestion of 6 documents


Ingesting:   0%|          | 0/6 [00:00<?, ?file/s]

2026-06-03 22:32:29,392 - ragwire.core.pipeline - INFO - Processed ../data/finance_data/apple 10-k 2024.pdf: 53 chunks


Ingesting:  17%|█▋        | 1/6 [03:06<15:30, 186.11s/file]

2026-06-03 22:35:15,838 - ragwire.core.pipeline - INFO - Processed ../data/finance_data/amazon 10k 2025.pdf: 38 chunks


...

Ingesting:  83%|████████▎ | 5/6 [14:26<02:53, 173.35s/file]

2026-06-03 22:46:33,068 - ragwire.core.pipeline - INFO - Processed ../data/finance_data/amazon 10-k 2024.pdf: 39 chunks


Ingesting: 100%|██████████| 6/6 [17:09<00:00, 171.63s/file]


2026-06-03 22:46:36,699 - ragwire.core.pipeline - INFO - Ingestion complete: 6/6 documents


{'total': 6,
 'processed': 6,
 'skipped': 0,
 'failed': 0,
 'chunks_created': 260,
 'errors': []}

## 4. Explore Metadata

RAGWire extracts company name, doc type, and fiscal year during ingestion. Let's inspect what's stored.

In [4]:
rag.discover_metadata_fields()

['source',
 'file_name',
 'file_type',
 'file_hash',
 'chunk_id',
 'chunk_hash',
 'chunk_index',
 'total_chunks',
 'created_at',
 'company_name',
 'doc_type',
 'fiscal_year',
 'fiscal_quarter']

In [5]:
rag.filter_fields

['company_name', 'doc_type', 'fiscal_year', 'fiscal_quarter']

## 5. Manual Metadata Filters

The simple agent above can mix up companies when all three are in the same collection.
Filters let us pin retrieval to a specific company, year, or doc type.

In [14]:
query = "what is apple's revenue in 2025?"
results = rag.retrieve(query=query)

2026-06-04 00:19:09,657 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: what is apple's revenue in 2025?...


In [15]:
results

[Document(metadata={'source': '../data/finance_data/Apple_10k_2025.pdf', 'file_name': 'Apple_10k_2025.pdf', 'file_type': 'pdf', 'file_hash': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab', 'chunk_id': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab_17', 'chunk_hash': '6f8d154ee3123334dcf4f277a15aaf1d235ed46a0a4fa4a3adcf44b3ade8c2b6', 'chunk_index': 17, 'total_chunks': 35, 'created_at': '2026-06-03T17:29:16.637684+00:00', 'company_name': 'apple inc.', 'doc_type': '10-k', 'fiscal_year': 2025, 'fiscal_quarter': None, '_id': '2233e39d-249c-4dd9-8feb-978aabb48eb0', '_collection_name': 'finance-rag-qdrant'}, page_content='Recently Adopted Accounting Pronouncements\n\nSegment Reporting\n\nBeginning  with  the  2025  annual  reporting  period,  the  Company  adopted  the  FASB’s  ASU  No.  2023-07,  Segment  Reporting  (Topic  280):  Improvements  to\nReportable Segment Disclosures (“ASU 2023-07”), which requires the Company to disclose segment expenses th

In [16]:
results = rag.retrieve(query=query, filters={'company_name':'apple inc.'})
results

2026-06-04 00:19:12,880 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: what is apple's revenue in 2025?...


[Document(metadata={'source': '../data/finance_data/Apple_10k_2025.pdf', 'file_name': 'Apple_10k_2025.pdf', 'file_type': 'pdf', 'file_hash': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab', 'chunk_id': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab_17', 'chunk_hash': '6f8d154ee3123334dcf4f277a15aaf1d235ed46a0a4fa4a3adcf44b3ade8c2b6', 'chunk_index': 17, 'total_chunks': 35, 'created_at': '2026-06-03T17:29:16.637684+00:00', 'company_name': 'apple inc.', 'doc_type': '10-k', 'fiscal_year': 2025, 'fiscal_quarter': None, '_id': '2233e39d-249c-4dd9-8feb-978aabb48eb0', '_collection_name': 'finance-rag-qdrant'}, page_content='Recently Adopted Accounting Pronouncements\n\nSegment Reporting\n\nBeginning  with  the  2025  annual  reporting  period,  the  Company  adopted  the  FASB’s  ASU  No.  2023-07,  Segment  Reporting  (Topic  280):  Improvements  to\nReportable Segment Disclosures (“ASU 2023-07”), which requires the Company to disclose segment expenses th

In [17]:
query = 'what is revenue on Google?'
results = rag.retrieve(query=query, filters={'company_name':'alphabet inc.'})
results

2026-06-04 00:19:14,920 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: what is revenue on Google?...


[Document(metadata={'source': '../data/finance_data/GOOG-10-K-2025.pdf', 'file_name': 'GOOG-10-K-2025.pdf', 'file_type': 'pdf', 'file_hash': '9953e8d2e70b18c5bf0af3b2b2a1e96de0555f51cbb39a625497b434e19fb43a', 'chunk_id': '9953e8d2e70b18c5bf0af3b2b2a1e96de0555f51cbb39a625497b434e19fb43a_18', 'chunk_hash': '9f0bebd9dbd7cd196a91bcbcc5909442aa6b786e5d10a8ee1d2379be8cf0ddb7', 'chunk_index': 18, 'total_chunks': 46, 'created_at': '2026-06-03T17:31:46.343838+00:00', 'company_name': 'alphabet inc.', 'doc_type': '10-k', 'fiscal_year': 2025, 'fiscal_quarter': None, '_id': '811bd5b5-2ee7-494f-9475-9afc45130c4d', '_collection_name': 'finance-rag-qdrant'}, page_content='Revenues and Monetization Metrics\n\nWe generate revenues by delivering relevant, cost-effective online advertising; cloud-based solutions that provide\nenterprise  customers  of  all  sizes  with  infrastructure,  platform  services,  and  applications;  and  sales  of  other  products\nand  services,  such  as  fees  received  for 

In [18]:
query = 'what is revenue of Google in 2024?'
results = rag.retrieve(query=query, filters={'company_name':'alphabet inc.', 'fiscal_year': 2024})
results

2026-06-04 00:19:16,707 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: what is revenue of Google in 2024?...


[Document(metadata={'source': '../data/finance_data/google 10-k 2024.pdf', 'file_name': 'google 10-k 2024.pdf', 'file_type': 'pdf', 'file_hash': '46106605128d8b0dfc668e25b7cc70bdfc68e87fa8ba5d7aad8b0456665d3db9', 'chunk_id': '46106605128d8b0dfc668e25b7cc70bdfc68e87fa8ba5d7aad8b0456665d3db9_21', 'chunk_hash': 'a23dfe8c7c435cf720e373996d1c03327c5a9313bb0b3a448b004a6993ed514b', 'chunk_index': 21, 'total_chunks': 49, 'created_at': '2026-06-03T17:26:21.450028+00:00', 'company_name': 'alphabet inc.', 'doc_type': '10-k', 'fiscal_year': 2024, 'fiscal_quarter': None, '_id': 'f9cb041f-53a9-456f-89a6-4769aaae0239', '_collection_name': 'finance-rag-qdrant'}, page_content='direct response advertising products, both of which benefited from increased spending by our advertisers.\n\nGoogle Network\n\nGoogle Network revenues decreased $953 million from 2023 to 2024, primarily driven by a decrease in Google Ad Manager and AdMob\n\nrevenues. Additionally, Google Network revenues were adversely affected b

## 6. Auto-Filter

RAGWire can extract filters from the query automatically — no need to pass them manually.

In [19]:
rag._auto_filter

False

In [20]:
rag._auto_filter = True

In [21]:
query = "what is apple's revenue in 2025?"
results = rag.retrieve(query=query)
results

2026-06-04 00:19:57,928 - ragwire.core.pipeline - INFO - Auto-extracted filters from query: {'company_name': 'apple inc.', 'fiscal_year': 2025}
2026-06-04 00:19:58,944 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: what is apple's revenue in 2025?...


[Document(metadata={'source': '../data/finance_data/Apple_10k_2025.pdf', 'file_name': 'Apple_10k_2025.pdf', 'file_type': 'pdf', 'file_hash': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab', 'chunk_id': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab_17', 'chunk_hash': '6f8d154ee3123334dcf4f277a15aaf1d235ed46a0a4fa4a3adcf44b3ade8c2b6', 'chunk_index': 17, 'total_chunks': 35, 'created_at': '2026-06-03T17:29:16.637684+00:00', 'company_name': 'apple inc.', 'doc_type': '10-k', 'fiscal_year': 2025, 'fiscal_quarter': None, '_id': '2233e39d-249c-4dd9-8feb-978aabb48eb0', '_collection_name': 'finance-rag-qdrant'}, page_content='Recently Adopted Accounting Pronouncements\n\nSegment Reporting\n\nBeginning  with  the  2025  annual  reporting  period,  the  Company  adopted  the  FASB’s  ASU  No.  2023-07,  Segment  Reporting  (Topic  280):  Improvements  to\nReportable Segment Disclosures (“ASU 2023-07”), which requires the Company to disclose segment expenses th

In [22]:
query = "what is google's revenue in 2024?"
results = rag.retrieve(query=query)
results

2026-06-04 00:20:21,128 - ragwire.core.pipeline - INFO - Auto-extracted filters from query: {'company_name': 'alphabet inc.', 'fiscal_year': 2024}
2026-06-04 00:20:22,271 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: what is google's revenue in 2024?...


[Document(metadata={'source': '../data/finance_data/google 10-k 2024.pdf', 'file_name': 'google 10-k 2024.pdf', 'file_type': 'pdf', 'file_hash': '46106605128d8b0dfc668e25b7cc70bdfc68e87fa8ba5d7aad8b0456665d3db9', 'chunk_id': '46106605128d8b0dfc668e25b7cc70bdfc68e87fa8ba5d7aad8b0456665d3db9_21', 'chunk_hash': 'a23dfe8c7c435cf720e373996d1c03327c5a9313bb0b3a448b004a6993ed514b', 'chunk_index': 21, 'total_chunks': 49, 'created_at': '2026-06-03T17:26:21.450028+00:00', 'company_name': 'alphabet inc.', 'doc_type': '10-k', 'fiscal_year': 2024, 'fiscal_quarter': None, '_id': 'f9cb041f-53a9-456f-89a6-4769aaae0239', '_collection_name': 'finance-rag-qdrant'}, page_content='direct response advertising products, both of which benefited from increased spending by our advertisers.\n\nGoogle Network\n\nGoogle Network revenues decreased $953 million from 2023 to 2024, primarily driven by a decrease in Google Ad Manager and AdMob\n\nrevenues. Additionally, Google Network revenues were adversely affected b

In [24]:
rag._auto_filter = False